# Entrenamiento YOLOv8 para deteccion de placas

Este cuaderno prepara el dataset, entrena varios modelos YOLOv8, muestra las graficas de Colab, evalua los resultados y conserva los tres mejores pesos.

Puedes montar Google Drive y guardar alli los pesos generados.

In [ ]:
# Instala las dependencias en Google Colab
%pip install -q ultralytics easyocr opencv-python-headless pillow pandas matplotlib seaborn
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuracion de rutas

Colab trabaja en una maquina temporal. Estas rutas separan los datos originales, el dataset convertido, las ejecuciones y los modelos finales:

- `PROJECT_ROOT`: carpeta local del proyecto dentro de la sesion de Colab.
- `DATA_ROOT`: imagenes y etiquetas originales (`models/Data`). No se modifican.
- `DATASET_ROOT`: copia generada con etiquetas YOLO normalizadas y carpetas `train`/`val`.
- `RUNS_ROOT`: ejecuciones y checkpoints (`last.pt`/`best.pt`) en Google Drive, para sobrevivir a una desconexion.
- `OUTPUT_ROOT`: los tres mejores modelos y `search_results.json`, tambien en Drive.

Deja `REBUILD_DATASET = False` para reutilizar el dataset ya construido. Solo cambia a `True` si cambias las imagenes o etiquetas.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/Placas')
DATA_ROOT = PROJECT_ROOT / 'models' / 'Data'
DATASET_ROOT = PROJECT_ROOT / 'models' / 'yolo_dataset_final_holdout'
FINAL_TEST_ROOT = PROJECT_ROOT / 'models' / 'final_test_holdout'
OUTPUT_ROOT = Path('/content/drive/MyDrive/Placas/top_models')
RUNS_ROOT = Path('/content/drive/MyDrive/Placas/runs_colab')
WEIGHTS = 'yolov8n.pt'
DEVICE = '0'
SEED = 42
TRIALS = 3
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 2
PATIENCE = 10
FINAL_TEST_RATIO = 0.20
INNER_VAL_RATIO = 0.20
REBUILD_DATASET = False

assert (DATA_ROOT / 'images').is_dir(), f'No existe: {DATA_ROOT / "images"}'
assert (DATA_ROOT / 'labels').is_dir(), f'No existe: {DATA_ROOT / "labels"}'
PROJECT_ROOT.joinpath('models').mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Datos originales: {DATA_ROOT}')
print(f'Dataset de entrenamiento/validacion: {DATASET_ROOT}')
print(f'Datos reservados para evaluacion final: {FINAL_TEST_ROOT}')
print(f'Checkpoints persistentes: {RUNS_ROOT}')
print(f'Mejores modelos: {OUTPUT_ROOT}')

## 2. Division y revision del dataset

Se reserva aleatoriamente el 20% de las imagenes como evaluacion final. Esas imagenes no se copian al dataset de YOLO y no se usan para entrenar, validar, elegir hiperparametros ni aplicar early stopping.

El 80% restante se divide en 80% entrenamiento y 20% validacion interna. Por tanto, sobre el total queda:

- `64%`: entrenamiento.
- `16%`: validacion interna para early stopping y seleccion durante las pruebas.
- `20%`: evaluacion final, tocado solo despues de seleccionar el modelo.

La semilla hace que la reserva sea aleatoria pero reproducible. Si `REBUILD_DATASET = False`, una reserva ya creada se conserva para no cambiar la comparacion entre modelos. YOLO baraja los ejemplos de entrenamiento en cada epoca; la validacion interna permanece fija, que es lo correcto para que early stopping sea comparable. Cambiar tambien la validacion en cada epoca produciria una metrica ruidosa y podria introducir fuga de informacion.

In [ ]:
from collections import Counter
import json
import random
import shutil

sys.path.insert(0, str(PROJECT_ROOT / 'models'))
from ia import build_dataset

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_MANIFEST = FINAL_TEST_ROOT / 'split_manifest.json'
MODEL_DATA_ROOT = PROJECT_ROOT / 'models' / 'training_data'

source_images = sorted(
    path for path in (DATA_ROOT / 'images').iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
)
label_paths = {path.stem: path for path in (DATA_ROOT / 'labels').glob('*.txt')}
paired_images = [path for path in source_images if path.stem in label_paths]
if len(paired_images) != len(source_images):
    print(f'Pares validos: {len(paired_images)}; imagenes sin etiqueta: {len(source_images) - len(paired_images)}')

if REBUILD_DATASET or not SPLIT_MANIFEST.exists():
    shuffled = paired_images.copy()
    random.Random(SEED).shuffle(shuffled)
    final_test_count = max(1, round(len(shuffled) * FINAL_TEST_RATIO))
    final_test_names = {path.name for path in shuffled[:final_test_count]}
    SPLIT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    SPLIT_MANIFEST.write_text(json.dumps({
        'seed': SEED,
        'final_test_ratio': FINAL_TEST_RATIO,
        'final_test_images': sorted(final_test_names),
    }, indent=2))
else:
    final_test_names = set(json.loads(SPLIT_MANIFEST.read_text())['final_test_images'])

training_images = [path for path in paired_images if path.name not in final_test_names]
if REBUILD_DATASET or not MODEL_DATA_ROOT.exists():
    if MODEL_DATA_ROOT.exists():
        shutil.rmtree(MODEL_DATA_ROOT)
    (MODEL_DATA_ROOT / 'images').mkdir(parents=True, exist_ok=True)
    (MODEL_DATA_ROOT / 'labels').mkdir(parents=True, exist_ok=True)
    for image_path in training_images:
        shutil.copy2(image_path, MODEL_DATA_ROOT / 'images' / image_path.name)
        shutil.copy2(label_paths[image_path.stem], MODEL_DATA_ROOT / 'labels' / f'{image_path.stem}.txt')

if REBUILD_DATASET or not FINAL_TEST_ROOT.exists():
    if FINAL_TEST_ROOT.exists():
        shutil.rmtree(FINAL_TEST_ROOT)
    (FINAL_TEST_ROOT / 'images').mkdir(parents=True, exist_ok=True)
    (FINAL_TEST_ROOT / 'labels').mkdir(parents=True, exist_ok=True)
    for image_path in paired_images:
        if image_path.name in final_test_names:
            shutil.copy2(image_path, FINAL_TEST_ROOT / 'images' / image_path.name)
            shutil.copy2(label_paths[image_path.stem], FINAL_TEST_ROOT / 'labels' / f'{image_path.stem}.txt')

DATASET_ROOT, CLASS_NAMES = build_dataset(
    MODEL_DATA_ROOT, DATASET_ROOT, val_ratio=INNER_VAL_RATIO, seed=SEED, force=REBUILD_DATASET
)
print('Clases:', CLASS_NAMES)
print('Dataset YAML:', DATASET_ROOT / 'dataset.yaml')
print(f'Total de imagenes con etiqueta: {len(paired_images)}')
print(f'Entrenamiento + validacion interna: {len(training_images)} ({len(training_images) / len(paired_images):.1%})')
print(f'Evaluacion final aislada: {len(final_test_names)} ({len(final_test_names) / len(paired_images):.1%})')

size_counts = Counter()
for image_path in source_images:
    with Image.open(image_path) as image:
        size_counts[image.size] += 1
print(f'Dimensiones distintas: {len(size_counts)}')
print('Dimensiones mas frecuentes:', size_counts.most_common(10))
if len(size_counts) == 1:
    print('Todas las imagenes tienen el mismo tamano.')
else:
    print('Las imagenes no tienen todas el mismo tamano; YOLO las redimensionara a IMAGE_SIZE durante el entrenamiento.')

train_images = list((DATASET_ROOT / 'images' / 'train').glob('*'))
val_images = list((DATASET_ROOT / 'images' / 'val').glob('*'))
print(f'Division interna: train={len(train_images)} ({len(train_images) / (len(train_images) + len(val_images)):.1%}), val={len(val_images)} ({len(val_images) / (len(train_images) + len(val_images)):.1%})')

label_files = list((DATASET_ROOT / 'labels').rglob('*.txt'))
class_counts = {}
for label_file in label_files:
    for line in label_file.read_text().splitlines():
        if line.strip():
            class_id = int(line.split()[0])
            class_counts[class_id] = class_counts.get(class_id, 0) + 1
counts = pd.Series(class_counts).sort_index()
counts.index = [CLASS_NAMES[int(index)] for index in counts.index]
counts.plot(kind='bar', title='Objetos por clase en entrenamiento y validacion', ylabel='Cantidad')
plt.tight_layout()
plt.show()

In [ ]:
# Muestra imagenes y sus cajas para detectar errores de etiquetado
import cv2
import random

def draw_labels(image_path, label_path):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue
        _, center_x, center_y, box_width, box_height = map(float, values)
        x1 = int((center_x - box_width / 2) * width)
        y1 = int((center_y - box_height / 2) * height)
        x2 = int((center_x + box_width / 2) * width)
        y2 = int((center_y + box_height / 2) * height)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return image

images = list((DATASET_ROOT / 'images' / 'train').glob('*'))
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, image_path in zip(axes.flat, random.Random(SEED).sample(images, min(6, len(images)))):
    axis.imshow(draw_labels(image_path, DATASET_ROOT / 'labels' / 'train' / f'{image_path.stem}.txt'))
    axis.set_title(image_path.name)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 3. Busqueda aleatoria y entrenamiento reanudable

Se prueban configuraciones con `AdamW`, augmentations, `patience=PATIENCE` y early stopping. Si la metrica no mejora durante `PATIENCE` epocas, Ultralytics detiene esa prueba y conserva `best.pt`.

Cada prueba guarda checkpoints periodicamente en Google Drive. Si Colab se desconecta, vuelve a ejecutar esta celda: cuando encuentre `last.pt`, continua desde los pesos y la epoca guardados. Los resultados terminados se registran en `search_results.json` y no se repiten.

La primera prueba parte de `yolov8n.pt`. Las pruebas siguientes se inicializan con uno de los tres mejores modelos disponibles hasta ese momento, por lo que sus pesos y bias aprendidos si se usan como punto de partida. `TRIALS`, `EPOCHS`, `BATCH_SIZE` y `PATIENCE` son parametros ajustables segun el tiempo de GPU disponible.

In [ ]:
from ultralytics import YOLO
import json
import random
import shutil

SEARCH_SPACE = {
    'lr0': [0.0005, 0.001, 0.005],
    'lrf': [0.01, 0.05, 0.1],
    'weight_decay': [0.0001, 0.0005, 0.001],
    'hsv_h': [0.01, 0.02], 'hsv_s': [0.5, 0.7],
    'hsv_v': [0.3, 0.5], 'degrees': [0.0, 5.0],
    'scale': [0.3, 0.5], 'fliplr': [0.0, 0.5],
    'mosaic': [0.5, 1.0], 'mixup': [0.0, 0.1],
    'dropout': [0.0, 0.1],
}
RESULTS_FILE = OUTPUT_ROOT / 'search_results_final_holdout.json'
generator = random.Random(SEED)
if RESULTS_FILE.exists():
    results = json.loads(RESULTS_FILE.read_text())
else:
    results = []
completed_trials = {item['trial'] for item in results}

for trial in range(1, TRIALS + 1):
    if trial in completed_trials:
        print(f'Trial {trial}/{TRIALS} ya registrado; se omite.')
        continue

    run_name = f'random_trial_{trial:03d}'
    run_dir = RUNS_ROOT / run_name
    last_checkpoint = run_dir / 'weights' / 'last.pt'
    best_checkpoint = run_dir / 'weights' / 'best.pt'
    previous_best = sorted(results, key=lambda item: item['score'], reverse=True)[:3]
    initial_weights = WEIGHTS if not previous_best else previous_best[(trial - 1) % len(previous_best)]['weights']
    config = {name: generator.choice(values) for name, values in SEARCH_SPACE.items()}

    if last_checkpoint.exists():
        print(f'Reanudando {run_name} desde {last_checkpoint}')
        model = YOLO(str(last_checkpoint))
        model.train(resume=True)
        initialization = str(last_checkpoint)
    else:
        print(f'Iniciando {run_name} desde {initial_weights}')
        model = YOLO(initial_weights)
        model.train(
            data=str(DATASET_ROOT / 'dataset.yaml'), epochs=EPOCHS, imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE, device=DEVICE, workers=WORKERS, patience=PATIENCE,
            seed=SEED + trial, project=str(RUNS_ROOT), name=run_name, exist_ok=True,
            optimizer='AdamW', amp=True, save_period=5, **config
        )
        initialization = initial_weights

    if not best_checkpoint.exists():
        raise FileNotFoundError(f'No se encontro el mejor checkpoint: {best_checkpoint}')
    metrics = YOLO(str(best_checkpoint)).val(
        data=str(DATASET_ROOT / 'dataset.yaml'), split='val', imgsz=IMAGE_SIZE, device=DEVICE
    )
    score = float(metrics.box.map)
    results.append({
        'trial': trial,
        'score': score,
        'weights': str(best_checkpoint),
        'initialization': initialization,
        'hyperparameters': config,
    })
    results = sorted(results, key=lambda item: item['score'], reverse=True)
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f'Trial {trial}/{TRIALS}: mAP50-95 validacion={score:.5f}. Progreso guardado en Drive.')

results = sorted(results, key=lambda item: item['score'], reverse=True)
pd.DataFrame([{'trial': item['trial'], 'mAP50-95 validacion': item['score']} for item in results])

## 4. Graficas de entrenamiento

Cada ejecucion de Ultralytics genera `results.png`, `confusion_matrix.png`, curvas Precision-Recall y `results.csv`.

In [ ]:
best_trial = results[0]
best_run = RUNS_ROOT / f"random_trial_{best_trial['trial']:03d}"
plot_files = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, filename in zip(axes.flat, plot_files):
    path = best_run / filename
    if path.exists():
        axis.imshow(Image.open(path))
        axis.set_title(filename)
    else:
        axis.text(0.5, 0.5, f'No generado: {filename}', ha='center')
    axis.axis('off')
plt.tight_layout()
plt.show()

csv_path = best_run / 'results.csv'
if csv_path.exists():
    history = pd.read_csv(csv_path)
    history.columns = history.columns.str.strip()
    display(history.tail())
    history.plot(x='epoch', y=[column for column in history.columns if 'loss' in column], figsize=(12, 5), title='Perdidas por epoca')
    plt.grid()
    plt.show()

## 5. Guardar los tres mejores modelos

Los modelos se ordenan usando solamente la `mAP50-95` de la validacion interna. Los archivos `.pt` contienen la arquitectura, los pesos y los bias aprendidos. Se copian los tres mejores y sus configuraciones; la evaluacion final sobre el 20% aislado se hace en la siguiente celda, despues de seleccionar el ganador.

In [ ]:
for rank, item in enumerate(results[:3], start=1):
    destination = OUTPUT_ROOT / f'model_{rank}'
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(item['weights'], destination / 'best.pt')
    metadata = {
        'rank': rank, 'trial': item['trial'], 'metric': 'mAP50-95',
        'score': item['score'], 'hyperparameters': item['hyperparameters'],
        'weights_file': 'best.pt',
    }
    (destination / 'configuration.txt').write_text(json.dumps(metadata, indent=2))
shutil.copy2(results[0]['weights'], PROJECT_ROOT / 'models' / 'best.pt')
print(f'Modelos guardados en {OUTPUT_ROOT}')
print('El mejor peso para la API:', PROJECT_ROOT / 'models' / 'best.pt')

## 6. Evaluacion final sobre datos nunca usados

Esta evaluacion se ejecuta despues de elegir el mejor modelo con la validacion interna. El conjunto final no participa en el entrenamiento, early stopping, busqueda de hiperparametros ni seleccion del top 3; su resultado es la estimacion final del rendimiento.

In [ ]:
best_model = YOLO(results[0]['weights'])
final_test_yaml = FINAL_TEST_ROOT / 'dataset.yaml'
final_test_yaml.write_text('\n'.join([
    f'path: {FINAL_TEST_ROOT.resolve().as_posix()}',
    'test: images',
    f'nc: {len(CLASS_NAMES)}',
    'names:',
    *[f'  {index}: {name}' for index, name in enumerate(CLASS_NAMES)],
]) + '\n')

final_metrics = best_model.val(
    data=str(final_test_yaml), split='test', imgsz=IMAGE_SIZE, device=DEVICE
)
print(f"Modelo ganador: trial {results[0]['trial']}")
print(f"mAP50-95 en evaluacion final: {float(final_metrics.box.map):.5f}")
print(f"mAP50 en evaluacion final: {float(final_metrics.box.map50):.5f}")

final_images = list((FINAL_TEST_ROOT / 'images').glob('*'))
sample_images = final_images[: min(6, len(final_images))]
predictions = best_model.predict(
    source=[str(path) for path in sample_images], conf=0.35,
    imgsz=IMAGE_SIZE, device=DEVICE
)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, prediction in zip(axes.flat, predictions):
    axis.imshow(prediction.plot()[..., ::-1])
    axis.axis('off')
plt.tight_layout()
plt.show()